# Homework 5 — Deploying a machine-learning model

You need internet access, a local Jupyter installation, and Docker Desktop (or Docker Engine) for Question 6. Start Docker before the container section. Hosted notebooks may not support Docker; run that section on a machine with Docker installed.

The notebook creates a `deployment_homework_2026` folder next to your notebook. The notebook kernel can use your existing Python: model code runs in a separate environment managed by `uv`, using the required Python 3.11.15. We do not add Jupyter dependencies to the homework lockfile.


## 1. Imports and workspace
These modules belong to Python's standard library. `Path` handles filenames; `subprocess` runs commands; `urllib` downloads files and sends HTTP requests. A separate folder keeps the homework files together.


In [1]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import time
import socket
from urllib.request import Request, urlopen
from urllib.error import URLError

PROJECT = Path.cwd() / "deployment_homework_2026"
PROJECT.mkdir(parents=True, exist_ok=True)
answers = {}  # Fill this dictionary as we complete each question.
print("Homework folder:", PROJECT)


Homework folder: /workspaces/MachineLearning_Zoomcamp2026/05_deployment/deployment_homework_2026


### A small command helper
A command is a list of arguments, which avoids shell quoting problems. `cwd` tells it which directory to run in. If a command fails, we display its output and stop that cell, so a failed setup cannot quietly produce misleading answers.


In [2]:
def run_command(arguments, *, cwd=PROJECT):
    result = subprocess.run(
        [str(argument) for argument in arguments],
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    result.check_returncode()
    return result.stdout


## 2. Download the reference release
We pin a Git commit so that rerunning this notebook downloads the same files even if the repository changes. The model is already trained; there is no need to download training data or retrain it for deployment.

Only load pickle files from a trusted source: unpickling can execute Python code. We use the course repository and check the published model checksum **before loading it**. A checksum verifies bytes, not whether an arbitrary source is trustworthy.


In [3]:
# This commit was resolved from the official homework directory when this notebook was made.
REFERENCE_COMMIT = "a395ca7ca995df92d8321daa017f3a1b4425a4a0"
BASE_URL = (
    "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/"
    f"{REFERENCE_COMMIT}/cohorts/2026/homework/05-deployment"
)
REFERENCE_FILES = [
    ".python-version", "pyproject.toml", "uv.lock", "pipeline.bin",
    "model.py", "predict.py", "feature_defaults.json", "model_metadata.json",
    "smoke_test.py", "q6_test.py", "Dockerfile",
]
print("Reference commit:", REFERENCE_COMMIT)


Reference commit: a395ca7ca995df92d8321daa017f3a1b4425a4a0


In [4]:
for filename in REFERENCE_FILES:
    destination = PROJECT / filename
    with urlopen(f"{BASE_URL}/{filename}", timeout=60) as response:
        contents = response.read()
    destination.write_bytes(contents)
    print(f"Downloaded {filename}: {len(contents):,} bytes")


Downloaded .python-version: 8 bytes
Downloaded pyproject.toml: 326 bytes
Downloaded uv.lock: 32,279 bytes
Downloaded pipeline.bin: 1,941 bytes
Downloaded model.py: 1,456 bytes
Downloaded predict.py: 1,426 bytes
Downloaded feature_defaults.json: 116 bytes
Downloaded model_metadata.json: 872 bytes
Downloaded smoke_test.py: 777 bytes
Downloaded q6_test.py: 828 bytes
Downloaded Dockerfile: 598 bytes


In [5]:
metadata = json.loads((PROJECT / "model_metadata.json").read_text())
expected_sha256 = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
actual_sha256 = hashlib.sha256((PROJECT / "pipeline.bin").read_bytes()).hexdigest()
assert actual_sha256 == expected_sha256 == metadata["model_sha256"]

# Missing-value defaults are also part of the reference release.
defaults_sha256 = hashlib.sha256((PROJECT / "feature_defaults.json").read_bytes()).hexdigest()
assert defaults_sha256 == metadata["feature_defaults_sha256"]
print("Model and feature-default checksums match.")


Model and feature-default checksums match.


## Question 1 — Check uv
`uv` manages Python environments and installs dependencies. This question is a setup note: record the version installed on your machine.

If the next cell cannot find it, run `%pip install uv` in a new notebook cell, then rerun the detection cell. The installer may require internet access. No specific local uv version is needed for this question; the Dockerfile pins its own version.


In [7]:
%pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 51.2 MB/s  0:00:00s eta 0:00:01

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
# Prefer uv on PATH; pip may install it beside the notebook's Python executable.
uv_path = shutil.which("uv")
if uv_path is None:
    candidate = Path(sys.executable).parent / ("uv.exe" if sys.platform == "win32" else "uv")
    if candidate.exists():
        uv_path = str(candidate)
if uv_path is None:
    raise RuntimeError("Install uv with %pip install uv, then rerun this cell.")
UV = uv_path
answers["Q1 (setup note)"] = run_command([UV, "--version"]).strip()


uv 0.13.5 (x86_64-unknown-linux-gnu)



### Create the locked environment
`uv sync --locked` checks and uses the existing lockfile. It must not regenerate it. The `.python-version` file selects Python 3.11.15; uv can download that interpreter if necessary. This first run may take a few minutes.


In [9]:
print("Requested Python:", (PROJECT / ".python-version").read_text().strip())
lock_before = hashlib.sha256((PROJECT / "uv.lock").read_bytes()).hexdigest()
run_command([UV, "sync", "--locked"])
assert hashlib.sha256((PROJECT / "uv.lock").read_bytes()).hexdigest() == lock_before


Requested Python: 3.11.15
Using CPython 3.11.15
Creating virtual environment at: .venv
Resolved 19 packages in 0.85ms
Prepared 18 packages in 719ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 18 packages in 1.07s
 + annotated-types==0.8.0
 + anyio==4.15.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + fastapi==0.119.1
 + h11==0.16.0
 + idna==3.19
 + joblib==1.6.0
 + numpy==2.3.3
 + pydantic==2.13.5
 + pydantic-core==2.46.5
 + scikit-learn==1.7.2
 + scipy==1.17.1
 + starlette==0.48.0
 + threadpoolctl==3.6.0
 + typing-extensions==4.16.0
 + typing-inspection==0.4.4
 + uvicorn==0.38.0



In [10]:
# Always use this interpreter for model code, rather than the Jupyter kernel.
MODEL_PYTHON = PROJECT / ".venv" / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
run_command([MODEL_PYTHON, "-c", "import platform; print(platform.python_version()); assert platform.python_version() == '3.11.15'"])
run_command([MODEL_PYTHON, "smoke_test.py"])


3.11.15

{'conversion_probability': 0.769799, 'conversion': True}



"{'conversion_probability': 0.769799, 'conversion': True}\n"

## Question 2 — Inspect the locked Scikit-Learn version
The project file declares dependencies; the lockfile resolves the environment. We read both with Python's TOML parser and confirm that the installed package matches. `tomllib` is available in the reference Python 3.11 environment, even if your notebook kernel is older.


In [11]:
print((PROJECT / "pyproject.toml").read_text())


[project]
name = "mlzoomcamp-2026-deployment"
version = "0.1.0"
description = "Reference deployment project for ML Zoomcamp 2026"
requires-python = ">=3.11,<3.12"
dependencies = [
    "fastapi==0.119.1",
    "numpy==2.3.3",
    "scikit-learn==1.7.2",
    "uvicorn==0.38.0",
]

[tool.uv]
exclude-newer = "2026-09-14T23:59:59Z"



In [12]:
inspect_environment = """
import json
import tomllib
from pathlib import Path
import sklearn

project = tomllib.loads(Path('pyproject.toml').read_text())
lock = tomllib.loads(Path('uv.lock').read_text())
locked = next(package['version'] for package in lock['package'] if package['name'] == 'scikit-learn')
declared = next(item for item in project['project']['dependencies'] if item.startswith('scikit-learn=='))
assert declared == 'scikit-learn==' + locked
assert sklearn.__version__ == locked
print(json.dumps({'declared': declared, 'locked': locked, 'installed': sklearn.__version__}))
"""
environment = json.loads(run_command([MODEL_PYTHON, "-c", inspect_environment]))
answers["Q2"] = environment["locked"]


{"declared": "scikit-learn==1.7.2", "locked": "1.7.2", "installed": "1.7.2"}



## Question 3 — Load the model and predict
A pipeline combines preprocessing and prediction. Here, `DictVectorizer` turns a dictionary into numeric features and logistic regression estimates class probabilities.

`predict_proba` takes a **batch** of records, so even one record is placed in a list. Its output has one row per record and one column per class. We inspect `classes_` to find the column for conversion (`1`), rather than assuming the column order.


In [13]:
lead_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}
(PROJECT / "lead_q3.json").write_text(json.dumps(lead_q3, indent=2))
lead_q3


{'lead_source': 'paid_ads',
 'industry': 'technology',
 'employment_status': 'employed',
 'location': 'north_america',
 'number_of_courses_viewed': 2,
 'annual_income': 79276.0,
 'interaction_count': 4,
 'lead_score': 0.41}

### Write the requested Python script
This cell saves a real `.py` file. The next cell runs it in the locked environment. `normalize_record` applies the same feature handling used by the API, including training-time defaults for missing values. Our lead already supplies every feature.


In [14]:
q3_script = '''
import hashlib
import json
import pickle
from pathlib import Path
from model import normalize_record

# Check the artifact again immediately before unpickling.
model_bytes = Path("pipeline.bin").read_bytes()
expected = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
assert hashlib.sha256(model_bytes).hexdigest() == expected
pipeline = pickle.loads(model_bytes)

lead = json.loads(Path("lead_q3.json").read_text())
record = normalize_record(lead)
classes = list(pipeline.classes_)
positive_column = classes.index(1)
probabilities = pipeline.predict_proba([record])
probability = float(probabilities[0, positive_column])
assert 0 <= probability <= 1
print(json.dumps({
    "conversion_probability": probability,
    "answer_to_three_decimals": f"{probability:.3f}",
    "class_order": [int(value) for value in classes],
    "pipeline_steps": list(pipeline.named_steps),
}))
'''
(PROJECT / "q3_predict.py").write_text(q3_script)
print(q3_script)



import hashlib
import json
import pickle
from pathlib import Path
from model import normalize_record

# Check the artifact again immediately before unpickling.
model_bytes = Path("pipeline.bin").read_bytes()
expected = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
assert hashlib.sha256(model_bytes).hexdigest() == expected
pipeline = pickle.loads(model_bytes)

lead = json.loads(Path("lead_q3.json").read_text())
record = normalize_record(lead)
classes = list(pipeline.classes_)
positive_column = classes.index(1)
probabilities = pipeline.predict_proba([record])
probability = float(probabilities[0, positive_column])
assert 0 <= probability <= 1
print(json.dumps({
    "conversion_probability": probability,
    "answer_to_three_decimals": f"{probability:.3f}",
    "class_order": [int(value) for value in classes],
    "pipeline_steps": list(pipeline.named_steps),
}))



In [15]:
q3_result = json.loads(run_command([MODEL_PYTHON, "q3_predict.py"]))
answers["Q3"] = q3_result["answer_to_three_decimals"]
print("Q3 answer:", answers["Q3"])


{"conversion_probability": 0.5329944186432507, "answer_to_three_decimals": "0.533", "class_order": [0, 1], "pipeline_steps": ["dictvectorizer", "logisticregression"]}

Q3 answer: 0.533


## Question 4 — Serve the model over HTTP
The supplied FastAPI application loads the model and exposes two routes:

- `GET /health` checks the service and reports artifact checksums.
- `POST /predict` accepts a JSON lead and returns a probability and a boolean decision.

Read the implementation below. The decision threshold is 0.5. A probability and a yes/no decision are different outputs.


In [16]:
print((PROJECT / "predict.py").read_text())


from __future__ import annotations

import hashlib
import os
import pickle
from pathlib import Path
from typing import Any

from fastapi import FastAPI
from pydantic import BaseModel

from model import DEFAULTS_PATH, normalize_record


MODEL_PATH = Path(os.getenv("MODEL_PATH", Path(__file__).with_name("pipeline.bin")))
with MODEL_PATH.open("rb") as model_file:
    pipeline = pickle.load(model_file)
MODEL_SHA256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
FEATURE_DEFAULTS_SHA256 = hashlib.sha256(DEFAULTS_PATH.read_bytes()).hexdigest()

app = FastAPI(title="ML Zoomcamp 2026 lead scoring")


class Lead(BaseModel):
    lead_source: str | None = None
    industry: str | None = None
    employment_status: str | None = None
    location: str | None = None
    number_of_courses_viewed: int | None = None
    annual_income: float | None = None
    interaction_count: int | None = None
    lead_score: float | None = None


@app.get("/health")
def health() -> dict[str, Any]:
    return {


### HTTP helpers
We use the standard library, so no requests package is needed. JSON is encoded into bytes for transport. The readiness helper retries briefly because starting a server is asynchronous.


In [17]:
def get_json(url):
    with urlopen(url, timeout=5) as response:
        return json.load(response)


def post_json(url, payload):
    request = Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urlopen(request, timeout=10) as response:
        return json.load(response)


def wait_for_health(base_url, timeout=60):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            health = get_json(base_url + "/health")
            if health.get("status") == "ok":
                return health
        except (URLError, TimeoutError, OSError):
            pass  # A new server may not yet be listening.
        time.sleep(0.5)
    raise TimeoutError("Server did not become ready. Inspect its log below.")


def check_prediction(result):
    probability = result["conversion_probability"]
    assert isinstance(probability, (int, float)) and not isinstance(probability, bool)
    assert 0 <= probability <= 1
    assert isinstance(result["conversion"], bool)


def require_free_port(port):
    with socket.socket() as probe:
        try:
            probe.bind(("127.0.0.1", port))
        except OSError as error:
            raise RuntimeError(f"Port {port} is busy. Stop the earlier server before continuing.") from error


### Start the API without blocking Jupyter
A foreground server would keep a notebook cell running indefinitely. `Popen` starts it in the background and sends logs to a file. We bind to loopback for a local notebook; the Dockerfile uses `0.0.0.0` inside the container.

If you rerun this section, run its cleanup cell first.


In [18]:
BASE_API = "http://127.0.0.1:9696"
require_free_port(9696)
api_log_path = PROJECT / "api.log"
with api_log_path.open("w") as log:
    api_process = subprocess.Popen(
        [str(MODEL_PYTHON), "-m", "uvicorn", "predict:app", "--host", "127.0.0.1", "--port", "9696"],
        cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT,
    )
try:
    health = wait_for_health(BASE_API)
    assert api_process.poll() is None, "Our API process exited unexpectedly."
    assert health["model_sha256"] == expected_sha256
    assert health["feature_defaults_sha256"] == defaults_sha256
    print(health)
except Exception:
    print(api_log_path.read_text())
    raise


{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}


In [19]:
# Q4 uses a different lead from Q3; do not reuse lead_q3.
lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}
result_q4 = post_json(BASE_API + "/predict", lead_q4)
check_prediction(result_q4)
answers["Q4"] = f"{result_q4['conversion_probability']:.3f}"
print("Full API response:", result_q4)
print("Q4 answer:", answers["Q4"])


Full API response: {'conversion_probability': 0.769799, 'conversion': True}
Q4 answer: 0.770


### Stop the local API
Free port 9696 before starting Docker, so the container request cannot accidentally hit the local Python server. This only stops the process created by this notebook.


In [20]:
if "api_process" in globals() and api_process.poll() is None:
    api_process.terminate()
    try:
        api_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        api_process.kill()
        api_process.wait(timeout=5)
print("Local API stopped.")


Local API stopped.


## Question 5 — Read the container configuration
The `FROM` instruction declares the Python base image. A tag identifies a named version; the digest after `@sha256:` pins its exact image contents. This question asks for the **tag**, not the local image size.

The Dockerfile installs the locked dependencies, copies the frozen model and API, and starts Uvicorn on port 9696.


In [21]:
dockerfile = (PROJECT / "Dockerfile").read_text()
print(dockerfile)


FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3

COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/

ENV PATH="/code/.venv/bin:$PATH" \
    UV_COMPILE_BYTECODE=1 \
    UV_LINK_MODE=copy

WORKDIR /code
COPY pyproject.toml uv.lock .python-version ./
RUN uv sync --locked --no-dev

COPY model.py predict.py feature_defaults.json model_metadata.json pipeline.bin q6_test.py ./

EXPOSE 9696
ENTRYPOINT ["uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"]



In [22]:
from_line = next(line for line in dockerfile.splitlines() if line.startswith("FROM "))
base_image = from_line.split()[1]
answers["Q5"] = base_image.split("@")[0]
print("Q5 answer:", answers["Q5"])


Q5 answer: python:3.11.15-slim-bookworm


## Question 6 — Build and test the container
Start Docker Desktop or your Docker Engine first. The first build downloads images and may take several minutes. If Docker is unavailable in your notebook environment, move the notebook to a local machine with Docker; do not count a second local API call as a container test.

The container includes its own Python and packages. `-p 127.0.0.1:9696:9696` forwards local port 9696 to port 9696 inside the container.


In [23]:
DOCKER = shutil.which("docker")
if DOCKER is None:
    raise RuntimeError("Docker is not on PATH. Install/start Docker and restart Jupyter if needed.")
run_command([DOCKER, "version"])  # Checks both the client and the running engine.


Client:
 Version:           29.8.0-1
 API version:       1.56
 Go version:        go1.27.1
 Git commit:        88096ef00576baf72a9cb45caa45c0544c40e0a7
 Built:             Wed Sep  9 11:17:47 2026
 OS/Arch:           linux/amd64
 Context:           default

Server:
 Engine:
  Version:          29.8.0-1
  API version:      1.56 (minimum version 1.40)
  Go version:       go1.27.1
  Git commit:       3ce5872b7950c63ba2ffbc5123101019ff3e6682
  Built:            Wed Sep  9 11:21:29 2026
  OS/Arch:          linux/amd64
  Experimental:     false
 containerd:
  Version:          2.3.4-2
  GitCommit:        db8809540e1a7a9da5d518876894933ff55692ab
 runc:
  Version:          1.5.1-13
  GitCommit:        8f2685a471d3347a686ad3909783d8aafc6bb208
 docker-init:
  Version:          0.19.0
  GitCommit:        



'Client:\n Version:           29.8.0-1\n API version:       1.56\n Go version:        go1.27.1\n Git commit:        88096ef00576baf72a9cb45caa45c0544c40e0a7\n Built:             Wed Sep  9 11:17:47 2026\n OS/Arch:           linux/amd64\n Context:           default\n\nServer:\n Engine:\n  Version:          29.8.0-1\n  API version:      1.56 (minimum version 1.40)\n  Go version:       go1.27.1\n  Git commit:       3ce5872b7950c63ba2ffbc5123101019ff3e6682\n  Built:            Wed Sep  9 11:21:29 2026\n  OS/Arch:          linux/amd64\n  Experimental:     false\n containerd:\n  Version:          2.3.4-2\n  GitCommit:        db8809540e1a7a9da5d518876894933ff55692ab\n runc:\n  Version:          1.5.1-13\n  GitCommit:        8f2685a471d3347a686ad3909783d8aafc6bb208\n docker-init:\n  Version:          0.19.0\n  GitCommit:        \n'

In [24]:
# Keep notebook-generated files and .venv out of the Docker build context.
# This does not change the canonical Dockerfile or its locked dependencies.
(PROJECT / ".dockerignore").write_text(".venv/\n__pycache__/\n*.log\n.ipynb_checkpoints/\n")
IMAGE = "zoomcamp-model:2026-hw5"
run_command([DOCKER, "build", "-t", IMAGE, "."])


#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 637B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
#2 ...

#3 [auth] astral-sh/uv:pull token for ghcr.io
#3 DONE 0.0s

#4 [auth] library/python:pull token for registry-1.docker.io
#4 DONE 0.0s

#5 [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4
#5 ...

#2 [internal] load metadata for docker.io/library/python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
#2 DONE 1.0s

#5 [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4
#5 DONE 1.2s

#6 [internal] load .dockerignore
#6 transferring context: 86B done
#6 DONE 0.0s

#7 [interna

'#0 building with "default" instance using docker driver\n\n#1 [internal] load build definition from Dockerfile\n#1 transferring dockerfile: 637B done\n#1 DONE 0.0s\n\n#2 [internal] load metadata for docker.io/library/python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3\n#2 ...\n\n#3 [auth] astral-sh/uv:pull token for ghcr.io\n#3 DONE 0.0s\n\n#4 [auth] library/python:pull token for registry-1.docker.io\n#4 DONE 0.0s\n\n#5 [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4\n#5 ...\n\n#2 [internal] load metadata for docker.io/library/python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3\n#2 DONE 1.0s\n\n#5 [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4\n#5 DONE 1.2s\n\n#6 [internal] load .dockerignore\n#6 transferring context: 86B done

In [25]:
require_free_port(9696)
# -d means detached: Docker returns the ID while the service keeps running.
# --rm removes this container when it is stopped; the built image remains.
container_id = run_command([
    DOCKER, "run", "--rm", "-d",
    "-p", "127.0.0.1:9696:9696", IMAGE,
]).strip()
print("Started container:", container_id)


616e679929a54cc42e8a62c260ba6ddfb22202ad30f6f20e77ae846d47312b54

Started container: 616e679929a54cc42e8a62c260ba6ddfb22202ad30f6f20e77ae846d47312b54


In [26]:
try:
    container_health = wait_for_health(BASE_API)
    assert container_health["status"] == "ok"
    assert container_health["model_sha256"] == metadata["model_sha256"]
    assert container_health["feature_defaults_sha256"] == metadata["feature_defaults_sha256"]
    print("Container health:", container_health)
except Exception:
    run_command([DOCKER, "logs", container_id])
    raise


Container health: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}


In [27]:
# Send the exact Q4 payload, now to the Docker-hosted service.
result_q6 = post_json(BASE_API + "/predict", lead_q4)
check_prediction(result_q6)
assert abs(result_q6["conversion_probability"] - result_q4["conversion_probability"]) < 1e-6
assert result_q6["conversion"] == result_q4["conversion"]
answers["Q6"] = f"{result_q6['conversion_probability']:.3f}"
print("Container response:", result_q6)
print("Q6 answer:", answers["Q6"])


Container response: {'conversion_probability': 0.769799, 'conversion': True}
Q6 answer: 0.770


### Run the course smoke client too
This independent client sends the same lead and checks the response shape. Its Python process runs outside the container and talks to the container over HTTP.


In [28]:
run_command([MODEL_PYTHON, "q6_test.py", BASE_API + "/predict"])


{'conversion_probability': 0.769799, 'conversion': True}



"{'conversion_probability': 0.769799, 'conversion': True}\n"

### Stop the container
Run this cleanup cell even if an earlier container test failed. It targets only the container ID created above.


In [29]:
if "container_id" in globals() and container_id:
    run_command([DOCKER, "stop", container_id])
    container_id = None
print("Container cleanup complete.")


616e679929a54cc42e8a62c260ba6ddfb22202ad30f6f20e77ae846d47312b54

Container cleanup complete.
